# 🧬 LINCS L1000 w predykcji interakcji lek–białko: Kompletny przewodnik**Autor:** Kacper  **Cel:** Krok po kroku wyjaśnić czym jest zbiór LINCS L1000, jak go przetwarzamy, i jak łączymy z naszym głównym datasetem (BindingDB) do trenowania multimodalnych modeli DTI.---## Spis treści1. **Tło biologiczne** – czym jest ekspresja genów i dlaczego nas interesuje2. **Architektura zbioru LINCS L1000** – jakie pliki pobraliśmy i co zawierają3. **Faza eksploracyjna (EDA)** – co robił notebook `explore_lincs_l1000.ipynb`4. **Faza produkcyjna** – jak `prepare_lincs_profiles.py` przetwarza surowe dane5. **Integracja z pipeline'em DTI** – jak model konsumuje profile L10006. **Podsumowanie** – pełny schemat przepływu danych

---## 1. 🧫 Tło biologiczne: ekspresja genów i leki### Czym jest ekspresja genów?Każda komórka ludzkiego ciała zawiera ~20 000 genów. W danym momencie tylko część z nich jest "włączona" (produkuje białka), a reszta jest "wyłączona". Ten stan on/off nazywamy **ekspresją genów**.Można to porównać do ogromnego miksetra z 20 000 suwakami – każdy suwak odpowiada jednemu genowi i może być podkręcony (gen aktywny) lub ściszony (gen wyciszony).### Co się dzieje, gdy podamy komórce lek?Gdy lek wchodzi do komórki i wiąże się z białkiem docelowym (targetem), uruchamia kaskadę zmian:```Lek → wiąże się z białkiem X → białko X przestaje działać  → szlak sygnałowy A zostaje zablokowany    → gen G1 zostaje wyciszony (z-score = -2.5)    → gen G2 zostaje pobudzony (z-score = +3.1)    → gen G3 bez zmian (z-score ≈ 0.0)    → ... (978 genów zmienia się w określony sposób)```Ta unikalna "sygnatura" zmian genowych jest jak **biologiczny odcisk palca leku**. Dwa leki, które blokują to samo białko, wywołają bardzo podobne zmiany w ekspresji genów, nawet jeśli ich struktury chemiczne (SMILES) wyglądają zupełnie inaczej!### Czym jest Z-score?Z-score to miara statystyczna mówiąca, o ile odchyleń standardowych wartość odbiega od średniej grupy kontrolnej (komórek bez leku):| Z-score | Interpretacja ||---------|--------------|| 0.0     | Gen nie zmienił się po podaniu leku || +2.0    | Gen został silnie pobudzony (nadekspresja) || -2.0    | Gen został silnie wyciszony (represja) || +5.0    | Ekstremalnie silne pobudzenie genu |

---## 2. 📦 Architektura zbioru LINCS L1000### Skąd pochodzi zbiór?Projekt **LINCS** (Library of Integrated Network-based Cellular Signatures) został uruchomiony przez NIH (National Institutes of Health) w USA. W ramach projektu naukowcy z **Broad Institute** (MIT/Harvard) stworzyli bazę danych **L1000**, w której:- Wzięli **~20 000 związków chemicznych** (leków i kandydatów na leki)- Podali je komórkom z **różnych linii nowotworowych** (A375, A549, HCC515, HEPG2, PC3, VCAP...)- W **różnych dawkach** (0.04µM, 0.12µM, 0.37µM, 1.11µM, 3.33µM, 10µM)- Na **różne czasy** (6h, 24h, 48h, 72h...)- I zmierzyli **zmiany ekspresji 978 kluczowych genów** (landmark genes)### Dlaczego tylko 978 genów, a nie wszystkie ~20 000?To genialne uproszczenie! Geny w komórce nie działają niezależnie — tworzą sieci regulacyjne. Naukowcy odkryli, że wystarczy fizycznie zmierzyć zaledwie **978 "genów-strażników" (landmark genes)**, a zachowanie pozostałych ~19 000 można wiarygodnie wyliczyć matematycznie (inference).Dzięki temu koszt jednego eksperymentu spadł z ~$500 do ~$5, co umożliwiło profilowanie milionów kombinacji lek × linia komórkowa × dawka × czas.### Jakie pliki pobraliśmy z serwera Broad Institute?```~/├── level5_beta_trt_cp_n720216x12328.gctx   ← 📊 Główna macierz (HDF5), ~35 GB│                                               720,216 eksperymentów × 12,328 genów│                                               Wartości = z-score'y (float32)│├── siginfo_beta.txt                         ← 📋 Katalog eksperymentów (~465 MB)│                                               Każdy wiersz = jeden eksperyment (sig_id)│                                               Kolumny: pert_id, dawka, czas, linia komórkowa...│├── geneinfo_beta.txt                        ← 🧬 Katalog genów (~50 KB)│                                               Które geny to "landmark" (978 fizycznie zmierzonych)│                                               Które to "inferred" (wyliczone matematycznie)│└── compoundinfo_beta.txt                    ← 💊 Katalog związków chemicznych                                                pert_id, nazwa leku, SMILES, MOA, target...```

### Relacje między plikamiKluczowe jest zrozumienie hierarchii identyfikatorów:```pert_id (np. "BRD-K12345678")  = jeden związek chemiczny (np. Ibuprofen)  │  ├── sig_id #1: A375,  10µM, 24h  ← eksperyment na raku skóry  ├── sig_id #2: A549,  10µM, 24h  ← eksperyment na raku płuc  ├── sig_id #3: HEPG2, 10µM, 24h  ← eksperyment na raku wątroby  ├── sig_id #4: PC3,   10µM, 24h  ← eksperyment na raku prostaty  ├── sig_id #5: A549,   1µM,  6h  ← ten sam rak płuc, inna dawka i czas  └── ...```Jeden lek (`pert_id`) → wiele eksperymentów (`sig_id`) → każdy to wiersz w macierzy `.gctx`Plik `siginfo_beta.txt` to "książka telefoniczna", która mapuje `sig_id` → `pert_id` (i dodaje informacje o dawce, czasie, linii komórkowej itp.).

---## 3. 🔬 Faza eksploracyjna: notebook `explore_lincs_l1000.ipynb`Zanim zaczniemy przetwarzać gigabajty danych, musimy odpowiedzieć na fundamentalne pytania:- Czy nasze leki z BindingDB w ogóle występują w bazie LINCS?- Ile ich jest?- Jak wyglądają surowe dane w macierzy GCTX?Notebook `explore_lincs_l1000.ipynb` odpowiadał właśnie na te pytania. Przejdźmy przez niego krok po kroku.

### 3.1 Compound Info – metadane związków chemicznych (Sekcja 1 notebooka)Pierwszy krok to wczytanie katalogu związków chemicznych z LINCS (`compoundinfo_beta.txt`).

In [ ]:
# Tak wyglądał kod w notebooku:# compounds = pl.read_csv("compoundinfo_beta.txt", separator="\t")# # Wynik pokazał:# Shape: (39321, ~15 kolumn)# Kolumny: pert_id, cmap_name, canonical_smiles, inchi_key, target, moa, ...## Kluczowe odkrycia:# - 39,321 związków w katalogu LINCS# - 33,531 z nich ma zapisany wzór SMILES (canonical_smiles)# - ~8,000 ma przypisany mechanizm działania (MOA)# - ~6,000 ma przypisany target (białko docelowe)print("Compound Info zawiera informacje o ~39,000 związkach chemicznych")print("Najważniejsza kolumna: 'canonical_smiles' – pozwala połączyć LINCS z BindingDB")print("Druga kluczowa kolumna: 'pert_id' – unikalny ID związku w systemie LINCS")

### 3.2 Gene Info – metadane genów (Sekcja 2 notebooka)Następnie notebook eksplorował plik z informacjami o genach.

In [ ]:
# W notebooku:# genes = pl.read_csv("geneinfo_beta.txt", separator="\t")## Kluczowe odkrycia:# - 12,328 genów w pliku# - Kolumna 'feature_space' dzieli je na:#     - "landmark"  → 978 genów (fizycznie zmierzonych)#     - "inferred"  → ~10,174 genów (wyliczonych matematycznie)#     - "best inferred" → ~1,176 genów (najlepiej wyliczonych)print("=== Podział genów w L1000 ===")print(f"  Landmark (zmierzonych fizycznie):  978")print(f"  Best inferred (wyliczonych):      ~1,176")  print(f"  Inferred (wyliczonych):           ~10,174")print(f"  RAZEM:                             12,328")print()print("My używamy TYLKO 978 landmark genes — to najtwardsze dane pomiarowe.")print("Reszta jest wyliczona matematycznie i może wprowadzać szum.")

### 3.3 Sig Info – katalog eksperymentów (Sekcja 3 notebooka)To największy plik metadanych (~465 MB). Każdy wiersz to jeden eksperyment.

In [ ]:
# W notebooku:# siginfo = pl.read_csv("siginfo_beta.txt", separator="\t")## Wyniki:# - ~1,201,944 sygnatur (eksperymentów)# - Każda sygnatura ma:#   - sig_id:      unikalny ID eksperymentu#   - pert_id:     jaki lek podano#   - cell_iname:  na jakiej linii komórkowej#   - pert_time:   jak długo inkubowano (6h, 24h, 48h...)#   - nearest_dose: jaka dawka (w µM)#   - is_hiq:      czy to sygnatura wysokiej jakości (1/0)#   - pert_type:   typ perturbacji (trt_cp = compound treatment)print("=== Sig Info – katalog eksperymentów ===")print("Ponad 1.2 MILIONA eksperymentów w bazie!")print()print("Dla jednego leku (pert_id) może być np. 20-50 sygnatur,")print("bo testowano go na różnych liniach komórkowych, dawkach i czasach.")print()print("Kolumna 'is_hiq' (high quality) pozwala filtrować tylko wiarygodne wyniki.")

### 3.4 Macierz GCTX – serce danych (Sekcja 4 notebooka)Plik `.gctx` to format HDF5 zawierający gigantyczną macierz liczb zmiennoprzecinkowych.

In [ ]:
# W notebooku otwierano plik HDF5 i sprawdzano jego strukturę:# # with h5py.File("level5_beta_trt_cp_n720216x12328.gctx", "r") as f:#     matrix = f["0/DATA/0/matrix"]#     print(matrix.shape)  → (720216, 12328)## Struktura HDF5:#   📁 0/#     📁 DATA/0/#       📊 matrix: shape=(720216, 12328), dtype=float32#     📁 META/#       📁 COL/#         📊 id: shape=(12328,)  ← identyfikatory genów (kolumny)#       📁 ROW/#         📊 id: shape=(720216,) ← identyfikatory sygnatur (wiersze)print("=== Macierz GCTX ===")print("Wymiary: 720,216 wierszy × 12,328 kolumn")print()print("Każdy WIERSZ = jeden eksperyment (sig_id)")print("Każda KOLUMNA = jeden gen")print("Każda KOMÓRKA = z-score (jak bardzo ten lek zmienił ten gen)")print()print(f"Rozmiar w pamięci: {720216 * 12328 * 4 / 1e9:.1f} GB (float32)")print("Dlatego nie wczytujemy całości — tylko potrzebne wiersze i kolumny!")

### 3.5 Łączenie z BindingDB – kluczowy moment (Sekcja 5 notebooka)To najważniejsza część notebooka! Tu odkrywamy, ile naszych leków z BindingDB pokrywa się z bazą LINCS.**Strategia łączenia:**```BindingDB (nasz dataset treningowy DTI)     LINCS (dane o ekspresji genów)┌──────────────────────────────┐           ┌──────────────────────────────┐│ Ligand SMILES  │ Target      │           │ pert_id  │ canonical_smiles  ││ CC(=O)Oc1...   │ EGFR        │    JOIN   │ BRD-K123 │ CC(=O)Oc1...      ││ c1ccc(O)cc1    │ p53         │ ───────→  │ BRD-K456 │ c1ccc(O)cc1       ││ ...            │ ...         │  po SMILES│ ...      │ ...               │└──────────────────────────────┘           └──────────────────────────────┘```**Problem:** SMILES mogą być zapisane na różne sposoby!  Np. `OC1=CC=CC=C1` i `c1ccc(O)cc1` to ten sam fenol.**Rozwiązanie:** Kanonizacja RDKit — obie formy zamieniamy na jedną kanoniczną.

In [ ]:
# W notebooku kanonizowano SMILES z obu baz:## from rdkit import Chem# def canonicalize_smiles(smi):#     mol = Chem.MolFromSmiles(smi)#     return Chem.MolToSmiles(mol)  # zawsze ten sam, kanoiczny zapis## Następnie wykonywano inner join:# merged = bindingdb.join(cmap_compounds, on="canonical_smiles_rdkit", how="inner")## KLUCZOWE WYNIKI:# - Unikalne SMILES w CMap (LINCS):     ~33,531# - Unikalne SMILES w BindingDB:       ~236,379# - Wspólne SMILES (overlap):            ~1,686# - % CMap w BindingDB:                   ~5.0%print("=== Wynik łączenia BindingDB × LINCS ===")print(f"  SMILES w LINCS:           ~33,531")print(f"  SMILES w BindingDB:      ~236,379")print(f"  Wspólne (overlap):         ~1,686 unikalnych leków")print(f"  Merged wierszy (par):     ~45,967 (lek × białko)")print()print("To ~1,686 leków daje nam ~45,967 par lek-białko,")print("bo jeden lek może wiązać się z wieloma białkami!")

### 3.6 Podsumowanie notebooka i "Następne kroki"Na końcu notebook wypisał plan działania:> **Następne kroki:**> 1. Ekstrakcja profili L1000 z GCTX dla relevant_sig_ids> 2. Agregacja profili per pert_id (np. mediana)> 3. Dołączenie wektora L1000 jako dodatkowej modalności do modelu DTITe "następne kroki" to dokładnie to, co realizuje skrypt `prepare_lincs_profiles.py`.

---## 4. ⚙️ Faza produkcyjna: skrypt `prepare_lincs_profiles.py`Notebook udowodnił, że łączenie jest możliwe. Teraz skrypt realizuje pełny pipeline przetwarzania danych — od surowego pliku GCTX do gotowych tensorów PyTorch.### Schemat działania skryptu:```┌─────────────────────────┐     ┌─────────────────────────┐│ bindingdb_cmap_merged   │     │ siginfo_beta.txt        ││ _direct_SMILES.parquet  │     │ (katalog eksperymentów) ││                         │     │                         ││ ~45,967 wierszy         │     │ ~1.2M sygnatur          ││ z kolumną "pert_id"     │     │ pert_id → sig_id        │└────────────┬────────────┘     └────────────┬────────────┘             │                               │             └───────────┬───────────────────┘                         │ Krok 1-2: Które sig_id nas interesują?                         ▼              ┌──────────────────────┐              │ ~37,000 sygnatur     │              │ dla ~1,300 pert_ids  │              │ (is_hiq = 1)         │              └──────────┬───────────┘                         │    ┌────────────────────┤ Krok 3: Które kolumny wyciągnąć?    │                    │    ▼                    ▼┌──────────────┐   ┌────────────────────────────────────────┐│geneinfo_beta │   │ level5_beta_trt_cp_n720216x12328.gctx  ││  978 landmark│──▶│                                        ││  gene indices│   │ Macierz: 720K wierszy × 12K kolumn     │└──────────────┘   │                                        │                   │ Krok 4: Wyciągnij TYLKO:               │                   │   - wiersze = nasze sig_ids             │                   │   - kolumny = 978 landmark genes        │                   └──────────────────┬─────────────────────┘                                      │                                      ▼ Krok 5: Agregacja                              ┌──────────────────┐                              │ Dla każdego leku: │                              │ mediana z 20-50   │                              │ eksperymentów     │                              │ → 1 wektor (978,) │                              └────────┬─────────┘                                       │                        ┌──────────────┼──────────────┐                        ▼              ▼              ▼                 ┌─────────────┐ ┌──────────┐ ┌─────────────┐                 │lincs_profiles│ │smiles_to │ │lincs_balanced│                 │.pt          │ │_pert_id  │ │.parquet      │                 │             │ │.json     │ │              │                 │pert_id →    │ │SMILES →  │ │50/50 active/ │                 │tensor(978)  │ │pert_id   │ │inactive      │                 └─────────────┘ └──────────┘ └─────────────┘```

### Krok 1: Wczytanie merged datasetuSkrypt startuje od pliku `bindingdb_cmap_merged_direct_SMILES.parquet`, który notebook wyeksportował jako swój produkt końcowy.

In [ ]:
# === Krok 1 w prepare_lincs_profiles.py ===# # merged = pl.read_parquet("datasets/bindingdb_cmap_merged_direct_SMILES.parquet")## Ten plik to wynik joina z notebooka. Zawiera kolumny:# - Ligand SMILES          ← oryginalny wzór chemiczny z BindingDB# - canonical_smiles_rdkit ← skanonizowany SMILES (RDKit)# - Full_Protein_Sequence  ← sekwencja aminokwasowa białka docelowego# - pert_id                ← ID leku w systemie LINCS (np. "BRD-K12345678")# - is_active              ← czy lek wiąże się z białkiem (True/False)# - pKi                    ← siła wiązania (wartość liczbowa)# - cmap_name, target, moa ← metadane z LINCSprint("Krok 1: Wczytujemy merged dataset")print("  To produkt notebooka explore_lincs_l1000.ipynb")print("  Zawiera ~45,967 wierszy (par lek-białko)")print("  z ~1,686 unikalnymi lekami (pert_id)")

### Krok 2: Mapowanie pert_id → sig_id (które eksperymenty wyciągnąć?)Mamy ~1,686 leków, ale w macierzy GCTX jest 720,000 eksperymentów. Musimy wiedzieć, KTÓRE wiersze z macierzy odpowiadają NASZYM lekom.

In [ ]:
# === Krok 2 w prepare_lincs_profiles.py ===## siginfo = pl.read_csv("siginfo_beta.txt", separator="\t")# target_pert_ids = set(merged["pert_id"].unique().to_list())# relevant_sigs = siginfo.filter(pl.col("pert_id").is_in(list(target_pert_ids)))## # Filtrujemy do sygnatur wysokiej jakości# relevant_sigs = relevant_sigs.filter(pl.col("is_hiq") == 1)## # Budujemy mapowanie: pert_id → [lista sig_ids]# pert_to_sigs = {}# for row in relevant_sigs.iter_rows():#     pert_to_sigs.setdefault(row["pert_id"], []).append(row["sig_id"])print("Krok 2: Budujemy mapowanie pert_id → sig_ids")print()print("Przykład:")print("  BRD-K12345678 (Ibuprofen) → [")print("    'REP.A001_A375_24H_X1_B22_DMSO_PLATE1',   # rak skóry, 24h")print("    'REP.A002_A549_24H_X1_B22_DMSO_PLATE2',   # rak płuc, 24h")print("    'REP.A003_HEPG2_6H_X1_B22_DMSO_PLATE3',   # rak wątroby, 6h")print("    ...  (może być 20-50 takich eksperymentów)")print("  ]")print()print("Filtrujemy is_hiq=1, bo chcemy tylko wiarygodne eksperymenty.")print("Niska jakość = za mało komórek przeżyło, artefakty techniczne itp.")

### Krok 3: Identyfikacja 978 landmark genes w macierzy GCTXMacierz GCTX ma 12,328 kolumn (genów). My chcemy TYLKO 978 fizycznie zmierzonych.

In [ ]:
# === Krok 3 w prepare_lincs_profiles.py ===## genes = pl.read_csv("geneinfo_beta.txt", separator="\t")# landmark_gene_ids = set(#     genes.filter(pl.col("feature_space") == "landmark")["gene_id"]#     .cast(pl.Utf8).to_list()# )## # Otwieramy GCTX i sprawdzamy, które kolumny odpowiadają naszym genom# with h5py.File("level5.gctx", "r") as f:#     gctx_gene_ids = [x.decode() for x in f["0/META/ROW/id"][:]]## landmark_gene_indices = [#     i for i, gid in enumerate(gctx_gene_ids) #     if gid in landmark_gene_ids# ]# # Wynik: lista ~978 indeksów, np. [0, 3, 7, 12, 15, ...]print("Krok 3: Znajdujemy indeksy 978 landmark genes w macierzy")print()print("Macierz GCTX ma 12,328 kolumn (genów)")print("Kolumna 0 = gen #5720 (ACE2)")print("Kolumna 1 = gen #7850 (TP53)  ← landmark!")print("Kolumna 2 = gen #3920 (FOXP1) ← inferred (wyliczony)")print("Kolumna 3 = gen #1290 (BRCA1) ← landmark!")print("...")print()print("Wynik: lista indeksów, np. [1, 3, 8, 12, ...] — 978 pozycji")print("Teraz wiemy KTÓRE kolumny macierzy wyciągnąć.")

### Krok 4: Ekstrakcja profili z gigantycznego pliku GCTXTo najcięższa obliczeniowo część. Otwieramy plik HDF5 i wyciągamy z niego tylko potrzebne dane — wiersz po wierszu, w partiach (batchach).

In [ ]:
# === Krok 4 w prepare_lincs_profiles.py ===## with h5py.File("level5.gctx", "r") as f:#     matrix = f["0/DATA/0/matrix"]   # nie wczytuje do RAM!#     #     for row_idx, sig_id in sorted_pairs:#         # Wyciąga JEDEN wiersz, TYLKO kolumny landmark genes#         row_data = matrix[row_idx, gene_indices]  # shape: (978,)#         sig_profiles[sig_id] = row_data.astype(np.float32)## Wizualizacja tego co się dzieje:## Macierz GCTX (720,216 × 12,328):#              gen_0  gen_1  gen_2  gen_3  gen_4  gen_5 ... gen_12327# sig_0      [ 0.12  -1.34   0.55   2.10  -0.03   0.88 ...   1.23 ]# sig_1      [ 0.45   0.22  -0.11   0.03   1.55  -2.01 ...   0.44 ]# sig_2      [-1.20   0.67   0.33   1.80  -0.90   0.12 ...  -0.55 ]  ← nasz!# ...# sig_720215 [ 0.09  -0.45   0.78  -1.23   0.34   0.56 ...   0.11 ]#                      ↑              ↑                          #                  landmark       landmark    #                  (bierzemy!)    (bierzemy!)## Wynik dla sig_2: [-1.20, 2.10, ...] — wektor 978 wartości z-scoreprint("Krok 4: Ekstrakcja z pliku GCTX")print()print("h5py otwiera plik w trybie 'lazy' — nie wczytuje 35 GB do RAM!")print("Zamiast tego sięga bezpośrednio na dysk po konkretny wiersz.")print()print("Sortujemy wiersze rosnąco po indeksie, bo HDF5 czyta szybciej")print("gdy sięga po dane 'po kolei' (sekwencyjny dostęp do dysku).")print()print("Wynik: słownik sig_profiles[sig_id] = np.array(978,)")

### Krok 5: Agregacja – z wielu eksperymentów robimy JEDEN wektor per lekTo kluczowy moment biologiczny! Jeden lek ma np. 30 różnych eksperymentów (różne nowotwory, dawki, czasy). Musimy je skondensować do jednego wektora.

In [ ]:
# === Krok 5 w prepare_lincs_profiles.py ===## for pert_id, sig_ids in pert_to_sigs.items():#     arrays = [sig_profiles[s] for s in sig_ids]#     stacked = np.stack(arrays, axis=0)   # shape: (n_sigs, 978)#     median_profile = np.median(stacked, axis=0)  # shape: (978,)#     pert_profiles[pert_id] = torch.tensor(median_profile)## Wizualizacja agregacji dla Ibuprofenu (BRD-K12345678):##   Eksperyment na raku skóry (A375):    [ 0.12, -1.34,  2.10, -0.03, ...]#   Eksperyment na raku płuc (A549):     [ 0.45,  0.22,  1.80,  0.15, ...]#   Eksperyment na raku wątroby (HEPG2): [-1.20,  0.67,  1.90, -0.10, ...]#   Eksperyment na raku prostaty (PC3):  [ 0.30, -0.45,  2.30,  0.05, ...]#                                          ────────────────────────────#   MEDIANA (konsensus):                 [ 0.21, -0.12,  2.00,  0.01, ...]## Dlaczego MEDIANA a nie ŚREDNIA?# - Mediana jest odporna na wartości odstające (outliers)# - Jeśli na jednej linii komórkowej lek zadziałał dziwnie#   (np. komórki umierały z innego powodu), mediana to zignoruje# - Średnia byłaby zniekształcona przez ten jeden zły eksperymentprint("Krok 5: Agregacja per pert_id (mediana)")print()print("Dlaczego mediana zamiast średniej?")print("  Średnia:  (0.12 + 0.45 + (-1.20) + 0.30) / 4 = -0.08")print("  Mediana:  sortujemy [-1.20, 0.12, 0.30, 0.45] → (0.12+0.30)/2 = 0.21")print()print("Mediana ignoruje tę jedną drastycznie ujemną wartość (-1.20)")print("z raka wątroby, gdzie lek mógł zadziałać nietypowo.")print()print("Wynik: słownik pert_profiles[pert_id] = torch.Tensor(978,)")

### Krok 6-7: Zapis plików wyjściowychSkrypt generuje trzy pliki, które model DTI konsumuje podczas treningu.

In [ ]:
# === Krok 6: Zapis profili ===# torch.save(pert_profiles, "datasets/lincs_profiles.pt")## Zawartość: {#   "BRD-K12345678": tensor([0.21, -0.12, 2.00, 0.01, ...]),  # 978 wartości#   "BRD-K87654321": tensor([1.30, -0.55, 0.10, -1.80, ...]), #   ...  # ~1,300 leków# }## === Krok 7: Zapis mapowania SMILES → pert_id ===# json.dump(smiles_pert_map, "datasets/smiles_to_pert_id.json")## Zawartość: {#   "CC(=O)Oc1ccccc1C(=O)O": "BRD-K12345678",  # aspiryna#   "CC(C)Cc1ccc(cc1)C(C)C(=O)O": "BRD-K87654321",  # ibuprofen#   ...  # ~1,800 mapowań (z duplikatami SMILES)# }## === Krok 8: Zbalansowany dataset ===# balanced.write_parquet("datasets/lincs_balanced.parquet")# # Filtrujemy merged do leków z profilami L1000, potem# balansujemy active/inactive do 50/50print("Krok 6-8: Trzy pliki wyjściowe")print()print("1. lincs_profiles.pt (5.3 MB)")print("   → słownik: pert_id → tensor(978)")print("   → ~1,300 leków, każdy jako wektor 978 z-score'ów")print()print("2. smiles_to_pert_id.json (168 KB)")print("   → słownik: SMILES → pert_id")print("   → 'ściągawka' pozwalająca zamienić wzór chemiczny na ID LINCS")print()print("3. lincs_balanced.parquet (738 KB)")print("   → zbalansowany dataset (50% active, 50% inactive)")print("   → ~27,498 wierszy gotowych do scaffold split i treningu")

---## 5. 🔗 Integracja z pipeline'em DTITeraz najciekawsza część: jak model "zjada" te dane podczas treningu?### Przepływ danych w czasie treningu:```                         ┌─────────────────────────┐                         │  lincs_balanced.parquet  │                         │  (SMILES, Protein, label)│                         └────────────┬────────────┘                                      │                                      ▼                              ┌──────────────┐                              │  DTIDataset   │                              │ __getitem__(i) │                              └───────┬──────┘                                      │ zwraca (SMILES, Protein_seq, label)                                      │                       ┌──────────────┴──────────────┐                       ▼                              ▼              ┌─────────────────┐            ┌─────────────────┐              │ SMILES Processors│            │ Protein Processor│              │ (dla każdego     │            │ (sekwencja → int)│              │  typu enkodera)  │            └─────────────────┘              └────────┬────────┘                       │        ┌──────────────┼──────────────────┐        ▼              ▼                  ▼┌──────────────┐ ┌───────────┐  ┌─────────────────┐│LincsProcessor│ │ FP Process│  │ChemBERT Process  ││              │ │           │  │                  ││SMILES        │ │SMILES     │  │SMILES            ││  ↓ json      │ │  ↓ RDKit  │  │  ↓ tokenizer     ││pert_id       │ │fingerprint│  │token_ids          ││  ↓ .pt cache │ │(1024-bit) │  │(from pretrained)  ││tensor(978)   │ │           │  │                   │└──────┬───────┘ └─────┬─────┘  └────────┬──────────┘       │               │                 │       ▼               ▼                 ▼┌──────────────┐ ┌───────────┐  ┌─────────────────┐│LincsEncoder  │ │FP MLP     │  │ChemBERT Encoder  ││(MLP: 978→128)│ │(1024→128) │  │(768→128)         │└──────┬───────┘ └─────┬─────┘  └────────┬──────────┘       │               │                 │       └───────────────┼─────────────────┘                       │ concatenate                       ▼              ┌─────────────────┐              │  Fusion MLP     │              │  (384 → 128 →   │              │   32 → 1)       │──── + Protein embedding              └─────────────────┘                       │                       ▼                   P(active)```

### Co robi `LincsProcessor`?Plik `processing/smiles/lincs_processor.py` to "tłumacz" między SMILES a profilem L1000.

In [ ]:
# === LincsProcessor – przetwarzanie SMILES → profil L1000 ===## class LincsProcessor:#     def __init__(self, cache_path, smiles_pert_map_path):#         self._profiles = torch.load("lincs_profiles.pt")      # pert_id → tensor#         self._smiles_to_pert = json.load("smiles_to_pert_id.json")  # SMILES → pert_id##     def process(self, smiles: str) -> torch.Tensor:#         # Krok 1: SMILES → pert_id (lookup w JSON)#         pert_id = self._smiles_to_pert.get(smiles)  # np. "BRD-K12345678"#         #         # Krok 2: pert_id → profil (lookup w .pt cache)#         profile = self._profiles.get(pert_id)  # tensor(978,)#         #         return profile  # wektor 978 z-score'ów## Cały "tłumacz" to dwa lookupy w słownikach — ZERO obliczeń!# Dlatego LINCS jest tak szybki w treningu.print("LincsProcessor – dwa lookupy w pamięci:")print()print("  Wejście: 'CC(=O)Oc1ccccc1C(=O)O'  (SMILES aspiryny)")print("      ↓ lookup w smiles_to_pert_id.json")print("  Pośredni: 'BRD-A12345678'  (pert_id)")print("      ↓ lookup w lincs_profiles.pt")print("  Wyjście: tensor([ 0.21, -0.12, 2.00, 0.01, ..., -0.55])  (978 wartości)")print()print("Czas: <0.001 ms per sample (natychmiastowy lookup w RAM)")

### Co robi `LincsEncoder`?Plik `encoders/smiles/lincs_encoder.py` to sieć neuronowa (MLP), która kompresuje 978-wymiarowy wektor do 128-wymiarowego embeddingu.

In [ ]:
# === LincsEncoder – MLP kompresujący profil L1000 ===## class LincsEncoder(nn.Module):#     def __init__(self, input_dim=978, hidden_dim=128, out_dim=128):#         self.mlp = nn.Sequential(#             nn.Linear(978, 128),     # 978 → 128 (kompresja)#             nn.BatchNorm1d(128),     # normalizacja#             nn.ReLU(),               # aktywacja#             nn.Dropout(0.5),         # regularyzacja#             nn.Linear(128, 128),     # 128 → 128 (projekcja)#             nn.BatchNorm1d(128),#             nn.ReLU()#         )## Wizualizacja:#   Wejście:  [z1, z2, z3, ..., z978]  ← surowe z-score'y 978 genów#      ↓  Linear(978, 128)   ← uczy się, które geny są ważne#      ↓  BatchNorm + ReLU   ← stabilizacja treningu#      ↓  Dropout(0.5)       ← zapobieganie overfittingowi#      ↓  Linear(128, 128)   ← dalsze przetwarzanie#   Wyjście: [e1, e2, ..., e128]  ← "esencja" biologicznego działania lekuprint("LincsEncoder – kompresja 978 → 128")print()print("Sieć UCZY SIĘ, które z 978 genów są najważniejsze")print("dla przewidywania interakcji lek-białko.")print()print("Np. może odkryć, że geny związane z apoptozą (śmiercią komórek)")print("są kluczowe dla przewidywania, czy lek wiąże się z kinazami,")print("podczas gdy geny metabolizmu lipidów są mniej istotne.")

---## 6. 📊 Podsumowanie – pełny schemat przepływu danych### Od surowych plików do predykcji:```FAZA 1: Eksploracja (notebook)══════════════════════════════  BindingDB (clean.parquet) ──┐                              ├─ inner join po canonical SMILES ──→ merged.parquet  CMap (compoundinfo) ────────┘                                     (~45,967 par)FAZA 2: Przetwarzanie (skrypt)══════════════════════════════  merged.parquet ─────────────┐  siginfo_beta.txt ───────────┤  geneinfo_beta.txt ──────────┤──→ prepare_lincs_profiles.py  level5.gctx (35 GB) ───────┘        │                                       ├──→ lincs_profiles.pt      (pert_id → 978-d tensor)                                       ├──→ smiles_to_pert_id.json (SMILES → pert_id)                                       └──→ lincs_balanced.parquet (zbalansowany dataset)FAZA 3: Trening modelu (main.py)══════════════════════════════  lincs_balanced.parquet ──→ DTIDataset ──→ DataLoader                                               │                                    ┌──────────┴──────────┐                                    ▼                      ▼                              SMILES branch          Protein branch                              (LincsProcessor        (ProteinProcessor                               + LincsEncoder)        + CNN/ESM2)                                    │                      │                                    └──────────┬───────────┘                                               ▼                                          Fusion MLP                                               │                                               ▼                                    P(lek wiąże się z białkiem)```### Kluczowe liczby:| Element | Wartość ||---------|---------|| Leków w LINCS | ~20,000 || Leków wspólnych z BindingDB | ~1,686 || Landmark genes (wymiar wektora) | 978 || Eksperymentów na lek (mediana) | ~20-50 || Par lek-białko (merged) | ~45,967 || Par po balansowaniu | ~27,498 || Rozmiar profilu per lek | 978 × float32 = 3.8 KB || Rozmiar pliku lincs_profiles.pt | 5.3 MB |

### Dlaczego to działa?1. **Komplementarność modalności:** SMILES mówi "jak lek WYGLĄDA",    a LINCS mówi "co lek ROBI w żywej komórce". To dwa zupełnie różne źródła informacji.2. **Scaffold Hopping:** Dwa leki o różnej strukturze chemicznej (różne SMILES/Fingerprinty),    ale trafiające w to samo białko, będą miały podobne profile L1000.    LINCS wykryje to, czego sama chemia nie widzi.3. **Pre-obliczone dane:** Profil L1000 to "gotowa wiedza" z prawdziwych eksperymentów biologicznych,    nie uczona od zera na naszym małym zbiorze. Model dostaje gotowy, bogaty sygnał biologiczny.4. **Szybkość:** Lookup w słowniku RAM zajmuje <0.001 ms.    Nie ma żadnych kosztownych obliczeń podczas treningu (w przeciwieństwie do GCN).